## Steps we will follow 
1. Preprocessing And Clustering
2. Train Test Split 
3. Bow,Tfidf,WOrd2vec
4. Train ML Algorithm 

In [1]:
import pandas as pd 

In [2]:
# Load the dataset

df = pd.read_csv("../kindlereview Dataset/all_kindle_review.csv")


In [3]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [4]:
df=df[['reviewText','rating']]

In [5]:
df.shape

(12000, 2)

In [6]:
# Missing values 
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [7]:
# Checking unique data
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [8]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

### Preprocessing Cleaning

In [9]:
# Positive 1 and negative review 0
df['rating']= df['rating'].apply(lambda x:0 if x<3 else 1)

In [10]:
df['rating'].unique()

array([1, 0])

In [11]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [12]:
# Lower the cases
df['reviewText'] = df['reviewText'].str.lower()

In [13]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [14]:
import re
import nltk 
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to C:\Users\Ayush
[nltk_data]     Kashyap\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
# Cleaining
# Removing special character
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub('[^a-zA-Z0-9]'," ",x))

# removing stopwords
df['reviewText'] = df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))



In [16]:
# Removing url
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?'," ",str(x)))

In [17]:
%pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [18]:
from bs4 import BeautifulSoup

In [19]:
# Remove html tags 
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x, 'lxml').get_text())

In [20]:
# Removing additional spaces

df['reviewText'] = df['reviewText'].apply(lambda x:" ".join(x.split()))

In [21]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four books expecting 34 con...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [22]:
# Applying lemmatization

from nltk.stem import WordNetLemmatizer

In [34]:
lemmatizer = WordNetLemmatizer()

In [35]:
def lemmatization_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [36]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatization_words(x))

In [37]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting 34 conc...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [38]:
# train test split 
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],
                                                df['rating'],
                                                test_size=0.20)

In [39]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow = bow.transform(X_test).toarray()

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf = tfidf.transform(X_test).toarray()

In [46]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow= GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf = GaussianNB().fit(X_train_tfidf,y_train)

In [45]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [47]:
y_pred_bow = nb_model_bow.predict(X_test_bow)
y_pred_tfidf = nb_model_tfidf.predict(X_test_tfidf)

In [49]:
print("BOW Accuracyt : ",accuracy_score(y_test,y_pred_bow))

BOW Accuracyt :  0.5716666666666667


In [50]:
confusion_matrix(y_test,y_pred_bow)

array([[565, 264],
       [764, 807]])

In [51]:
print("TFIDF Accuracyt : ",accuracy_score(y_test,y_pred_tfidf))

TFIDF Accuracyt :  0.5770833333333333


In [52]:
confusion_matrix(y_test,y_pred_tfidf)

array([[547, 282],
       [733, 838]])